In [1]:
from langchain_ollama import ChatOllama

# Best general model: strong chat, instruction following, and tool calling.
MODEL_QWEN3_8B = "qwen3:8b"
# Good general-purpose alternative for chat, planning, and explanations.
MODEL_LLAMA31_8B = "llama3.1:8b"
# Good conversational and writing model with balanced speed and quality.
MODEL_MISTRAL_7B = "mistral:7b"
# Fast lightweight general model for simple questions and low latency.
MODEL_QWEN25_3B = "qwen2.5:3b"
# Fast lightweight chat model for simple agents and everyday tasks.
MODEL_LLAMA32_3B = "llama3.2:3b"
# Best coding model in this collection: code generation, debugging, and APIs.
MODEL_QWEN25_CODER_7B = "qwen2.5-coder:7b"
# Best reasoning-focused model: math, logic, and multi-step analysis; slower.
MODEL_DEEPSEEK_R1_7B = "deepseek-r1:7b"
# Compact general model with good instruction following.
MODEL_GEMMA3_4B = "gemma3:4b"
# Small reasoning/instruction model; useful when conserving VRAM.
MODEL_PHI4_MINI = "phi4-mini:latest"
# Fastest chat/agent model here; lower quality on complex tasks.
MODEL_LLAMA32_1B = "llama3.2:1b"
# Small Qwen model for fast lightweight prompts and classification.
MODEL_QWEN25_15B = "qwen2.5:1.5b"
# Vision model for image and screenshot understanding; not text-only chat.
MODEL_LLAVA_7B = "llava:7b"
# Embedding model for RAG/vector search; do not use as a chat model.
MODEL_NOMIC_EMBED_TEXT = "nomic-embed-text:latest"

# Change this one value to switch the chat model used below.
MODEL_NAME = MODEL_LLAVA_7B
OLLAMA_URL = "http://127.0.0.1:11434"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    temperature=0,
)

print(f"Connected to Ollama. Using local model: {MODEL_NAME}")

/home/kanadmarick/Workspace/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connected to Ollama. Using local model: llava:7b


In [2]:
import os
from pathlib import Path
from typing import Annotated
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, interrupt
from typing_extensions import TypedDict
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
import fitz
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
for env_path in [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "Langgraph_tools" / "chatbot" / ".env",
]:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        break

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=OLLAMA_URL,
    temperature=0,
)
# Minimal smoke test: confirm the model is actually responding.
response = llm.invoke("Reply with one short sentence confirming the model is working.")
print(response.content)

/tmp/ipykernel_1197053/3940996141.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


 The model is working. 


LANGSMITH_PROJECT = "multimodalRAG"
langsmith_api_key = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")

if langsmith_api_key:
    os.environ["LANGSMITH_API_KEY"] = langsmith_api_key
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGSMITH_ENDPOINT"] = "https://apac.api.smith.langchain.com"
    os.environ["LANGCHAIN_ENDPOINT"] = "https://apac.api.smith.langchain.com"
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    from langchain_core.tracers import LangChainTracer

    langsmith_tracer = LangChainTracer(project_name=LANGSMITH_PROJECT)
    print(f"LangSmith tracing enabled for project: {LANGSMITH_PROJECT}")
else:
    print(
        "LangSmith tracing is disabled: set LANGSMITH_API_KEY in your local .env file."
    )

In [3]:

class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [4]:
## Clip Model
import os
from dotenv import load_dotenv

load_dotenv()

## Setup the environment
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
# Initialze CLIP model for unified embedding

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")  # Initialize your CLIP model here
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")  # Initialize your CLIP processor here
## clip_model.eval()  # Set the CLIP model to evaluation mode

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 21141.50it/s]


In [5]:
def embed_text(text: str):
    """Embed text using the CLIP text encoder."""
    device = next(clip_model.parameters()).device
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
        # Handle different output wrapper formats
        if hasattr(outputs, "text_embeds"):
            embeddings = outputs.text_embeds
        elif isinstance(outputs, torch.Tensor):
            embeddings = outputs
        else:
            embeddings = outputs[0]
            
        # Normalize embeddings to unit vector
        embeddings = embeddings / embeddings.norm(p=2, dim=-1, keepdim=True)
        
    return embeddings.squeeze().cpu().numpy()

In [6]:
## Process PDF

pdf_path="../pdf/image-doc.pdf"
doc=fitz.open(pdf_path)
# Storage for all th documents
all_docs = []
all_embeddings = []
image_data_store = {} # Store actual image data for LLM

#Text splitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)



In [7]:
doc

Document('../pdf/image-doc.pdf')

In [10]:
import torch
import numpy as np
from PIL import Image

def embed_image(image_data):
    """Embed image using CLIP model and return 1D numpy array."""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    else:
        image = image_data

    inputs = clip_processor(images=image, return_tensors="pt")
    device = next(clip_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clip_model.get_image_features(**inputs)
        if hasattr(outputs, "image_embeds"):
            embeddings = outputs.image_embeds
        elif isinstance(outputs, torch.Tensor):
            embeddings = outputs
        else:
            embeddings = outputs[0]

        embeddings = embeddings / embeddings.norm(p=2, dim=-1, keepdim=True)

    # .flatten() guarantees a 1D vector (e.g. shape (512,))
    return embeddings.squeeze().cpu().numpy().flatten().astype(np.float32)


def embed_text(text: str):
    """Embed text using CLIP text encoder and return 1D numpy array."""
    device = next(clip_model.parameters()).device
    inputs = clip_processor(text=[text], return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = clip_model.get_text_features(**inputs)
        if hasattr(outputs, "text_embeds"):
            embeddings = outputs.text_embeds
        elif isinstance(outputs, torch.Tensor):
            embeddings = outputs
        else:
            embeddings = outputs[0]

        embeddings = embeddings / embeddings.norm(p=2, dim=-1, keepdim=True)

    # .flatten() guarantees a 1D vector (e.g. shape (512,))
    return embeddings.squeeze().cpu().numpy().flatten().astype(np.float32)

In [15]:
import io
import base64
from PIL import Image
import torch
import numpy as np
import fitz  # PyMuPDF
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_core.embeddings import FakeEmbeddings

# -------------------------------------------------------------
# 1. Robust Embedding Helper Functions (Guaranteed 1D float32)
# -------------------------------------------------------------
def embed_text(text: str) -> np.ndarray:
    """Encodes text to a normalized 1D float32 vector of shape (512,)."""
    text_content = text.strip() if text and text.strip() else "empty"
    device = next(clip_model.parameters()).device
    
    inputs = clip_processor(
        text=[text_content], 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=77
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        if hasattr(features, "text_embeds"):
            features = features.text_embeds
        elif not isinstance(features, torch.Tensor):
            features = features[0]
            
        if features.ndim == 3:
            features = features.mean(dim=1)
            
        features = features / features.norm(p=2, dim=-1, keepdim=True)
        
    return features.squeeze().detach().cpu().numpy().flatten().astype(np.float32)


def embed_image(image_data) -> np.ndarray:
    """Encodes PIL image/path to a normalized 1D float32 vector of shape (512,)."""
    if isinstance(image_data, str):
        image = Image.open(image_data).convert("RGB")
    elif isinstance(image_data, Image.Image):
        image = image_data.convert("RGB")
    else:
        raise ValueError(f"Unsupported image input type: {type(image_data)}")

    device = next(clip_model.parameters()).device
    inputs = clip_processor(images=image, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        if hasattr(features, "image_embeds"):
            features = features.image_embeds
        elif not isinstance(features, torch.Tensor):
            features = features[0]
            
        if features.ndim == 3:
            features = features.mean(dim=1)
            
        features = features / features.norm(p=2, dim=-1, keepdim=True)

    return features.squeeze().detach().cpu().numpy().flatten().astype(np.float32)


# -------------------------------------------------------------
# 2. Extract and Embed from PDF
# -------------------------------------------------------------
all_docs = []
all_embeddings = []
image_data_store = {}

# Ensure PDF document handle is open
if "doc" not in locals() or doc.is_closed:
    doc = fitz.open(pdf_path)

for i, page in enumerate(doc):
    # A. Text Processing
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        chunks = text_splitter.split_documents([temp_doc])
        for chunk in chunks:
            all_docs.append(chunk)
            all_embeddings.append(embed_text(chunk.page_content))

    # B. Image Processing
    image_list = page.get_images(full=True)
    for img_index, img in enumerate(image_list):
        try:
            xref = img[0]
            if xref == 0:
                continue

            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_id = f"page_{i}_img_{img_index}"

            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            image_data_store[image_id] = img_base64

            all_embeddings.append(embed_image(pil_image))
            all_docs.append(Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            ))
        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")

# -------------------------------------------------------------
# 3. Stack Uniform Array & Build FAISS Index
# -------------------------------------------------------------
assert len(all_docs) > 0, "No documents or images found in the PDF."
assert len(all_docs) == len(all_embeddings), f"Mismatch: {len(all_docs)} docs vs {len(all_embeddings)} embeddings"

# np.vstack guarantees a 2D float32 matrix of shape (N, 512)
embeddings_array = np.vstack(all_embeddings).astype(np.float32)

vector_store = FAISS.from_embeddings(
    text_embeddings=[(d.page_content, emb) for d, emb in zip(all_docs, embeddings_array)],
    embedding=FakeEmbeddings(size=embeddings_array.shape[1]),
    metadatas=[d.metadata for d in all_docs]
)

print(f"Success! Indexed {vector_store.index.ntotal} items (embeddings shape: {embeddings_array.shape})")

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 1, the array at index 0 has size 512 and the array at index 1 has size 768

In [11]:
all_docs = []
all_embeddings = []
image_data_store = {}

for i, page in enumerate(doc):
    # -------------------------------------------------------------
    # 1. Process Text
    # -------------------------------------------------------------
    text = page.get_text()
    if text.strip():
        temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
        text_chunks = text_splitter.split_documents([temp_doc])
        for chunk in text_chunks:
            all_docs.append(chunk)
            all_embeddings.append(embed_text(chunk.page_content))

    # -------------------------------------------------------------
    # 2. Process Images
    # -------------------------------------------------------------
    image_list = page.get_images(full=True)
    for img_index, img in enumerate(image_list):
        try:
            xref = img[0]
            if xref == 0:
                continue

            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
            image_id = f"page_{i}_img_{img_index}"

            buffered = io.BytesIO()
            pil_image.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            image_data_store[image_id] = img_base64

            embedding = embed_image(pil_image)
            all_embeddings.append(embedding)

            image_doc = Document(
                page_content=f"[Image: {image_id}]",
                metadata={"page": i, "type": "image", "image_id": image_id}
            )
            all_docs.append(image_doc)

        except Exception as e:
            print(f"Error processing image {img_index} on page {i}: {e}")
            continue

print(f"Total embeddings created: {len(all_embeddings)}")
print(f"Total documents created: {len(all_docs)}")

Total embeddings created: 17
Total documents created: 17


In [12]:
len(all_embeddings)


17

In [13]:
len(all_docs)

17

In [14]:
from langchain_community.vectorstores import FAISS

embeddings_array = np.array(all_embeddings)

vector_store = FAISS.from_embeddings(
    text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
    embedding=None,
    metadatas=[doc.metadata for doc in all_docs]
)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (17,) + inhomogeneous part.

In [ ]:
## doc.close

In [ ]:
# What happens AFTER this output?
# How do all_embeddings and all_docs get populated?

print(f"all_embeddings length: {len(all_embeddings)}")
print(f"all_docs length: {len(all_docs)}")

# Also show me the type and shape
print(f"all_embeddings type: {type(all_embeddings)}")
if isinstance(all_embeddings, list) and len(all_embeddings) > 0:
    print(f"all_embeddings[0] shape: {np.array(all_embeddings[0]).shape}")

print(f"all_docs type: {type(all_docs)}")

In [ ]:
for idx, emb in enumerate(all_embeddings):
    arr = np.asarray(emb)
    print(f"Index {idx} ({all_docs[idx].metadata.get('type')}): shape={arr.shape}, type={type(emb)}")